# SpatialRoom3D: tái tạo không gian nội thất 3D từ ảnh / video, chạy trên Runpod

Notebook này nằm trong repo (`app/notebooks/runpod.ipynb`), dùng cho **pod mới gắn network volume mới (trống)**.

**Cách dùng:** tải file notebook này lên `/workspace` (kéo thả vào thanh file bên trái), mở bản ở `/workspace`, chạy từng ô bằng **Shift+Enter**. Không mở thẳng bản trong repo: Jupyter ghi kết quả chạy vào file, làm repo trên pod có thay đổi và ô 3 sẽ không cập nhật code. Chép lại khi cần: `cp /workspace/SpatialRoom3D/app/notebooks/runpod.ipynb /workspace/`.

**Bố cục trên volume:**
```
/workspace/
├── SpatialRoom3D/        code (repo Git, clone từ GitHub)
├── data/                 chỉ dữ liệu gốc (pipeline không ghi vào)
│   ├── custom_scenes/<cảnh>/        ảnh / video tự chụp
│   ├── benchmarks/{deep_blending,mipnerf360}/
│   └── downloads/                   file nén tải về
├── outputs/              mọi thứ sinh ra
│   ├── custom_scenes/<cảnh>/        colmap/ + các lượt train <chế_độ>_s<seed>/
│   ├── benchmarks/<benchmark>/      inria_b0/, gsplat/, gsplat_quick/
│   ├── checkpoints/                 PLY đã xuất để tải về / xem 3D
│   ├── logs/, reports/, jobs/       log, báo cáo, trạng thái việc chạy nền
├── miniforge3/, code/    env conda, mã nguồn 3DGS Inria và gsplat (ô 4 cài)
└── .ssh/                 khóa deploy chỉ đọc (ô 2); không bao giờ gửi thư mục này cho ai
```

| Lúc nào | Chạy các ô |
|---|---|
| Lần đầu trên pod mới, volume trống | 1 → 2 → 3 → 4 (theo dõi ở ô 5) → 6 |
| Baseline Deep Blending (tốn GPU, chỉ khi đã duyệt chi phí) | 7, theo dõi ở ô 5 |
| Cảnh tự chụp: thư mục ảnh hoặc video (tốn GPU) | 8 (`CHAY = True`), theo dõi ở ô 5, kết quả ở ô 9 |
| Xem mọi kết quả, đóng gói để tải về | 10 |
| Benchmark khác, thêm seed, đánh giá lại | 11 |
| Dừng mọi việc đang chạy | 12 (`DUNG=1`) |

- Việc dài đều chạy nền: đóng trình duyệt không sao, nhưng **pod vẫn tính tiền** tới khi Stop / Terminate.
- Mỗi lúc chỉ một việc dùng GPU; việc gửi sau tự xếp hàng.
- Xong việc: tải gói `ket_qua_*.tar.gz` (ô 10 in đường dẫn), rồi **Terminate pod**. Dữ liệu trên volume vẫn còn.
- Cùng các lệnh này bằng dòng lệnh: `cd /workspace/SpatialRoom3D && python3 -m app.cli --help` (hướng dẫn: `docs/USAGE_GUIDE.md`).

## 1. Kiểm tra máy

In [ ]:
%%bash
ver_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n 1)" = "$2" ]; }
gpu=$(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null | head -n 1)
cc=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -n 1 | tr -d ' ')
vram=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits 2>/dev/null | head -n 1 | tr -d ' ')
drv=$(nvidia-smi 2>/dev/null | grep -oE 'CUDA Version: *[0-9]+\.[0-9]+' | grep -oE '[0-9]+\.[0-9]+' | head -n 1)
nv=$(nvcc --version 2>/dev/null | grep -oE 'release [0-9]+\.[0-9]+' | grep -oE '[0-9]+\.[0-9]+')
gccv=$(gcc -dumpversion 2>/dev/null)
fst=$(findmnt -n -o FSTYPE --target /workspace 2>/dev/null)
echo "GPU:         ${gpu:-KHÔNG THẤY GPU} (compute capability ${cc:-?})"
echo "Driver CUDA: ${drv:-?} | nvcc: ${nv:-không có} | gcc: ${gccv:-?} | /workspace: ${fst:-?}"
echo "----------------------------------------"
ok=1
if [ -z "$drv" ]; then echo "✗ Không thấy GPU hoặc driver: deploy lại pod"; ok=0
elif ver_ge "$drv" 12.8; then echo "✓ Driver hỗ trợ CUDA $drv: dùng PyTorch cu128"
elif ver_ge "$drv" 12.6; then echo "! Driver chỉ hỗ trợ CUDA $drv: ở ô 4 đặt TORCH_CUDA_OVERRIDE=\"cu126\""
else echo "✗ Driver CUDA $drv quá cũ: Terminate rồi deploy máy khác"; ok=0; fi
case "$cc" in
  8.0|8.6|8.9|9.0) echo "✓ GPU có trong danh sách kiến trúc build sẵn (8.0/8.6/8.9/9.0)" ;;
  *) echo "✗ GPU compute capability ${cc:-?} không được hỗ trợ: dùng RTX 4090, A6000, A40, L40S, A100 hoặc H100"; ok=0 ;;
esac
if [ "${nv%%.*}" = "12" ]; then echo "✓ nvcc $nv"; else echo "✗ nvcc ${nv:-không có}: sai template, cần image runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04"; ok=0; fi
if [ -n "$gccv" ] && [ "${gccv%%.*}" -le 12 ]; then echo "✓ gcc $gccv"; else echo "✗ gcc ${gccv:-?}: cần gcc 12 trở xuống (template Ubuntu 22.04)"; ok=0; fi
if [ -n "$drv" ] && ! ver_ge "$drv" 12.9; then echo "! Driver CUDA $drv dưới 12.9: COLMAP dùng bản CPU (chậm hơn nhưng chạy được, tự chọn)"; fi
if [ "${vram:-0}" -ge 23000 ] 2>/dev/null; then echo "✓ VRAM ${vram} MiB"; else echo "! VRAM ${vram:-?} MiB dưới 24 GB: có thể thiếu bộ nhớ"; fi
case "$fst" in
  fuse*) echo "✓ /workspace là network volume" ;;
  *) echo "! /workspace là ổ '${fst:-?}', không phải network volume: Terminate pod sẽ mất dữ liệu" ;;
esac
if python3 -c 'import yaml' 2>/dev/null; then echo "✓ python3 có PyYAML (đọc configs/*.yaml)"
else echo "! python3 chưa có PyYAML: chạy 'python3 -m pip install pyyaml', hoặc chờ ô 4 cài xong env gs-gsplat (có sẵn PyYAML)"; fi
command -v git > /dev/null && command -v ssh-keygen > /dev/null && echo "✓ git, ssh có sẵn" || { echo "✗ thiếu git hoặc ssh"; ok=0; }
[ -d /workspace/miniforge3 ] && echo "! volume đã có /workspace/miniforge3 (không phải volume trống?)" || echo "✓ volume trống: ô 4 sẽ cài môi trường"
if [ "$ok" = 1 ]; then echo "=> ĐẠT: chạy tiếp ô 2"; else echo "=> CHƯA ĐẠT: sửa các dòng ✗ trước"; fi

## 2. Khóa deploy cho repo private (một lần cho mỗi volume)

Repo `SpatialRoom3D` là private. Pod đọc repo bằng một **deploy key chỉ đọc**: khóa chỉ mở được đúng repo này, không push được.

1. Chạy ô này: tạo khóa trong `/workspace/.ssh` (nếu chưa có) và in **khóa công khai**.
2. Trên GitHub: repo `SpatialRoom3D` → **Settings → Deploy keys → Add deploy key**, dán dòng khóa công khai, **không** tick "Allow write access".
3. Chạy lại ô này: dòng cuối phải là `✓ GitHub nhận khóa`.

Khóa riêng nằm trên volume (còn khi Terminate pod), không bao giờ được in hay gửi đi.

In [ ]:
%%bash
KEY=/workspace/.ssh/github_deploy
mkdir -p /workspace/.ssh && chmod 700 /workspace/.ssh
if [ -f "$KEY" ]; then
  echo "Khóa đã có: $KEY (không tạo lại)"
else
  ssh-keygen -t ed25519 -N "" -C "runpod-spatialroom3d" -f "$KEY" -q && echo "Đã tạo khóa mới: $KEY"
fi
chmod 600 "$KEY"
echo
echo "Khóa công khai, dán vào GitHub (SpatialRoom3D → Settings → Deploy keys → Add deploy key, KHÔNG tick 'Allow write access'):"
cat "$KEY.pub"
echo
SSH_CMD="ssh -i $KEY -o IdentitiesOnly=yes -o UserKnownHostsFile=/workspace/.ssh/known_hosts -o StrictHostKeyChecking=accept-new"
if $SSH_CMD -T git@github.com 2>&1 | grep -q "successfully authenticated"; then
  echo "✓ GitHub nhận khóa: chạy tiếp ô 3"
else
  echo "✗ GitHub chưa nhận khóa: thêm khóa công khai ở trên vào Deploy keys của repo SpatialRoom3D, rồi chạy lại ô này"
fi

## 3. Lấy code vào `/workspace/SpatialRoom3D`

- Chưa có: clone nhánh `BRANCH` từ GitHub bằng khóa của ô 2.
- Đã có: `git pull` (chỉ tiến lên, không ghi đè). Code trên pod có thay đổi chưa commit thì **không cập nhật** và in danh sách thay đổi; ô này không xóa hay cất gì.
- Không chạy khi đang có việc nền.

In [ ]:
%%bash
REPO_SSH="git@github.com:truong0809/SpatialRoom3D.git"
BRANCH="main"
CODE=/workspace/SpatialRoom3D
KEY=/workspace/.ssh/github_deploy
SSH_CMD="ssh -i $KEY -o IdentitiesOnly=yes -o UserKnownHostsFile=/workspace/.ssh/known_hosts -o StrictHostKeyChecking=accept-new"
if [ -f "$CODE/scripts/pod/job.sh" ] && bash "$CODE/scripts/pod/job.sh" busy; then
  echo "Đang có việc chạy nền: không đổi code lúc này (xem ô 5)"; exit 0
fi
if [ -d "$CODE/.git" ]; then
  changes="$(git -C "$CODE" status --porcelain)"
  if [ -n "$changes" ]; then
    echo "Code trên pod có thay đổi chưa commit: KHÔNG cập nhật (không xóa, không cất gì)."
    echo "$changes" | head -n 20
    echo "Xem: git -C $CODE status"
    exit 1
  fi
  GIT_SSH_COMMAND="$SSH_CMD" git -C "$CODE" pull --ff-only origin "$BRANCH" || { echo "Không cập nhật được (xem lỗi ở trên)"; exit 1; }
elif [ -e "$CODE" ]; then
  echo "$CODE đã có nhưng không phải repo Git: đổi tên hoặc xem lại, rồi chạy lại ô này"; exit 1
else
  GIT_SSH_COMMAND="$SSH_CMD" git clone -q -b "$BRANCH" "$REPO_SSH" "$CODE" \
    || { echo "Không clone được: chạy lại ô 2 (khóa deploy) và kiểm tra tên repo"; exit 1; }
  git -C "$CODE" config core.sshCommand "$SSH_CMD"
  echo "Đã clone $REPO_SSH (nhánh $BRANCH) vào $CODE"
fi
mkdir -p /workspace/data/custom_scenes /workspace/outputs
echo "Commit: $(git -C "$CODE" log --oneline -1)"

## 4. Cài môi trường (nền, không train)

`scripts/pod/run_all.sh --setup-only`: kiểm tra máy → Miniforge + env `gs-inria` (3DGS gốc của Inria) → env `gs-tools` (COLMAP, ffmpeg) → env `gs-gsplat` (kèm smoke test). Phiên bản ghim trong `configs/environment/`. **Không train gì.**

Lần đầu trên volume trống mất khoảng 80 phút (đo trên RTX 4090 ngày 17/10/2026: `gs-inria` 29 phút, `gs-gsplat` 32 phút, `gs-tools` 18 phút); pod vẫn tính tiền trong lúc cài. Chạy lại khi env đã có thì chỉ kiểm tra lại. Ô này xong ngay, việc chạy tiếp ở nền (theo dõi ở ô 5).

In [ ]:
%%bash
TORCH_CUDA_OVERRIDE=""   # "cu126" nếu ô 1 báo driver chỉ hỗ trợ CUDA 12.6
MAX_JOBS=""              # "4" nếu log báo Killed khi build gsplat (hết RAM)
cd /workspace/SpatialRoom3D || { echo "Chưa có /workspace/SpatialRoom3D: chạy ô 3"; exit 0; }
[ -n "$TORCH_CUDA_OVERRIDE" ] && export TORCH_CUDA_OVERRIDE
[ -n "$MAX_JOBS" ] && export MAX_JOBS
bash scripts/pod/run_all.sh --background --setup-only

## 5. Theo dõi

Cập nhật mỗi 15 giây: trạng thái các việc và cuối log của việc đang chạy; tự dừng khi không còn việc nào. Bấm nút ■ (Interrupt) để thôi theo dõi: việc nền vẫn chạy.

In [ ]:
import subprocess, time
from IPython.display import clear_output

JOB = "/workspace/SpatialRoom3D/scripts/pod/job.sh"

def job(*args):
    return subprocess.run(["bash", JOB, *args], capture_output=True, text=True, errors="replace")

while True:
    status = job("status").stdout
    running = [line.split()[0] for line in status.splitlines()
               if line and not line[0].isspace() and "ĐANG CHẠY" in line]
    tail = job("log", running[0], "20").stdout if running else ""
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), "(cập nhật mỗi 15 giây; bấm ■ để thôi theo dõi, việc nền vẫn chạy)\n")
    print(status)
    if running:
        print(f"---------- cuối log của {running[0]}")
        print(tail)
    if job("busy").returncode != 0:
        print("=> Không còn việc nào chạy hoặc xếp hàng.")
        break
    time.sleep(15)

## 6. Kiểm tra dữ liệu (chỉ đọc), tải Deep Blending (không cần GPU)

Theo `configs/*.yaml`: Deep Blending, Mip-NeRF 360 và cảnh tự chụp; thư mục gốc, từng cảnh có `images/` và `sparse/0/` không, có bao nhiêu lượt train.

Volume trống thì Deep Blending báo `THIẾU`: đặt `TAI_DEEP_BLENDING = True` để tải ngay (khoảng 650 MB, kiểm tra SHA-256, giải nén vào `data/benchmarks/deep_blending`), chạy song song được với ô 4 vì không dùng GPU. Không tải ở đây thì ô 7 tự tải.

Dữ liệu nằm chỗ khác (vd. Mip-NeRF 360 đặt tên `MipNeRF360`): tạo `/workspace/SpatialRoom3D/configs/local.yaml` với nội dung `datasets: {mipnerf360: {root: "${WS}/data/MipNeRF360"}}`, rồi chạy lại ô này.

In [ ]:
TAI_DEEP_BLENDING = False   # True: tải Deep Blending ngay (không cần GPU)

import subprocess
repo = "/workspace/SpatialRoom3D"
if TAI_DEEP_BLENDING:
    out = subprocess.run(["bash", f"{repo}/scripts/prepare_data.sh", "deep_blending"],
                         capture_output=True, text=True, errors="replace", cwd=repo)
    print(out.stdout[-3000:] + out.stderr[-2000:])
out = subprocess.run(["bash", f"{repo}/scripts/pod/check_datasets.sh"], capture_output=True, text=True,
                     errors="replace", cwd=repo)
print(out.stdout + out.stderr)

## 7. Baseline Deep Blending (tốn GPU: chỉ chạy khi đã duyệt chi phí)

`scripts/pod/run_all.sh` (đủ các bước): bỏ qua phần cài đặt đã xong → **chạy thử** gsplat 3.000 vòng → **B0** (3DGS gốc của Inria, đối chiếu số liệu công bố) → **gsplat** `default` rồi `mcmc` cùng số Gaussian, 30.000 vòng, seed 42 → bảng so sánh → gói kết quả.

Thời gian đo trên RTX 4090 ngày 17/10/2026 (code cũ, cùng tham số): chạy thử 17 phút, B0 khoảng 50 phút, gsplat 74 phút; tổng khoảng 2,4 giờ GPU. Lượt đã xong với cùng cấu hình tự bỏ qua khi chạy lại.

Đặt `CHAY = True` rồi chạy ô; theo dõi ở ô 5.

In [ ]:
CHAY = False        # True: chạy (đã duyệt chi phí GPU)
SEEDS = "42"        # vd. "42 43 44" cho bước gsplat (chạy thử chỉ dùng seed đầu)

import os, subprocess
repo = "/workspace/SpatialRoom3D"
if not CHAY:
    print("Chưa chạy: đặt CHAY = True khi đã duyệt chi phí GPU.")
else:
    out = subprocess.run(["bash", f"{repo}/scripts/pod/run_all.sh", "--background"], capture_output=True, text=True,
                         errors="replace", cwd=repo, env=dict(os.environ, SEEDS=SEEDS))
    print(out.stdout + out.stderr)

## 8. Cảnh tự chụp (thư mục ảnh hoặc video) → COLMAP → train `default` → train `mcmc`

**Ảnh chụp** (JPG/JPEG/PNG, ít nhất 10 ảnh, nên 30–100, cùng một căn phòng, cùng chiều máy, dùng file ảnh gốc của máy để giữ EXIF; cách chụp: `docs/data/capture_protocol.md` mục 8):
1. Trong thanh file, vào `/workspace/data/custom_scenes`, bấm nút thư mục mới (New Folder), đặt tên cảnh, ví dụ `phong1`.
2. Mở thư mục đó, bấm Upload và chọn **nhiều ảnh cùng lúc** (hoặc kéo thả nhiều file).
3. Đặt `SOURCE = "/workspace/data/custom_scenes/phong1"`.

**Video:** tạo thư mục như trên, tải video vào đó, đặt `SOURCE` là đường dẫn file video (vd. `/workspace/data/custom_scenes/phong2/phong2.mp4`).

Đặt `CHAY = True` rồi chạy ô (tốn GPU: COLMAP rồi hai lượt train, chạy nền). `SCENE` để trống thì lấy tên thư mục chứa ảnh / video. Kết quả: `/workspace/outputs/custom_scenes/<cảnh>/` (`colmap/` và các lượt train, mỗi lượt có ảnh test, video quỹ đạo và file PLY); thư mục ảnh gốc không bị sửa.

Một ảnh, dưới 10 ảnh, ảnh khác kích thước, hay tên cảnh đã dùng cho nguồn khác thì ô này báo lỗi ngay, không xếp hàng. COLMAP báo `FAIL` thì không train; lý do và cách chụp lại ở ô 9 (`gợi ý`, `góc nhìn`).

- Mỗi nguồn một tên cảnh (một job). Cảnh đã có COLMAP thì chỉ train (không đọc lại nguồn); `SOURCE = "-"` cũng là chỉ train (khi đó phải đặt `SCENE`).
- `REDO_COLMAP = True`: làm lại bước ảnh đầu vào + COLMAP; lượt train cũ được giữ dưới tên `*.prev_<thời gian>`.
- `RETRAIN = True`: train lại cả hai chế độ dù đã có kết quả.

In [ ]:
CHAY = False               # True: chạy (COLMAP + train default, mcmc; tốn GPU)
SOURCE = "/workspace/data/custom_scenes/phong1"  # thư mục ảnh hoặc file video; "-" = chỉ train cảnh đã có
SCENE = ""                 # để trống: tên thư mục chứa ảnh / video; chỉ chữ, số và . _ - (không dấu cách)
REDO_COLMAP = False        # True: làm lại bước ảnh đầu vào + COLMAP từ đầu
RETRAIN = False            # True: train lại mọi chế độ dù đã có kết quả
FRAMES_ARGS = ""           # tham số thêm, vd. "--target 300" (video) hoặc "--min-images 8" (ảnh)

import os, shlex, subprocess
if os.path.isdir(SOURCE):
    default = os.path.basename(SOURCE.rstrip("/"))
else:  # video: tên thư mục chứa nó; video đặt thẳng trong thư mục chung thì lấy tên file
    default = os.path.basename(os.path.dirname(os.path.abspath(SOURCE)))
    if default in ("custom_scenes", "data", "workspace", ""):
        default = os.path.splitext(os.path.basename(SOURCE))[0]
scene = SCENE.strip() or default
if SOURCE == "-" and not SCENE.strip():
    print("SOURCE = \"-\" (chỉ train) cần SCENE")
elif not CHAY:
    print(f"Chưa chạy cảnh {scene}: đặt CHAY = True (COLMAP + train, tốn GPU).")
else:
    cmd = ["bash", "/workspace/SpatialRoom3D/scripts/pod/run_scene.sh", SOURCE, scene, "--background"]
    if REDO_COLMAP:
        cmd.append("--redo-colmap")
    if RETRAIN:
        cmd.append("--retrain")
    if FRAMES_ARGS.strip():
        cmd += ["--", *shlex.split(FRAMES_ARGS)]
    print(f"cảnh: {scene}")
    out = subprocess.run(cmd, capture_output=True, text=True, errors="replace", cwd="/workspace/SpatialRoom3D")
    print(out.stdout + out.stderr)

## 9. Kết quả một cảnh, xuất PLY, render lại

Ô này in: ảnh đầu vào, kết quả COLMAP, gợi ý, độ phủ góc nhìn, các lượt train (PSNR / SSIM / LPIPS, số Gaussian), video quỹ đạo và file PLY.

- `XUAT_PLY = True`: đưa file PLY của lượt `RUN` vào `/workspace/outputs/checkpoints/custom_scene/<cảnh>/<lượt>.ply` để tải về và mở bằng trình xem 3D Gaussian Splatting (không cần GPU, không tốn thêm chỗ).
- `RENDER_LAI = True`: render lại ảnh test và video quỹ đạo `TRAJ` (`interp`, `ellipse`, `spiral`) từ checkpoint của lượt `RUN`, vào thư mục mới trong lượt đó (chạy nền, cần GPU; không train, không ghi đè lượt gốc).

In [ ]:
SCENE = "phong1"
RUN = "default_s42"      # lượt: <chế_độ>_s<seed>, vd. mcmc_s42
XUAT_PLY = False         # True: xuất PLY của lượt RUN (không cần GPU)
RENDER_LAI = False       # True: render lại ảnh test + video từ checkpoint (chạy nền, cần GPU)
TRAJ = "interp"          # quỹ đạo video: "interp", "ellipse" hoặc "spiral"

import os, subprocess
repo = "/workspace/SpatialRoom3D"
out = subprocess.run(["bash", f"{repo}/scripts/pod/results.sh", SCENE], capture_output=True, text=True,
                     errors="replace", cwd=repo)
print(out.stdout + out.stderr)
if XUAT_PLY:
    out = subprocess.run(["bash", f"{repo}/scripts/run_render.sh", "custom", SCENE, RUN], capture_output=True,
                         text=True, errors="replace", cwd=repo, env=dict(os.environ, RENDER="0"))
    print(out.stdout + out.stderr)
if RENDER_LAI:
    out = subprocess.run(["bash", f"{repo}/scripts/pod/job.sh", "start", f"render_{SCENE}_{RUN}", "--",
                          "bash", f"{repo}/scripts/run_render.sh", "custom", SCENE, RUN], capture_output=True,
                         text=True, errors="replace", cwd=repo, env=dict(os.environ, TRAJ=TRAJ))
    print(out.stdout + out.stderr)

## 10. Mọi kết quả (đọc file) và gói kết quả nhẹ để tải về (không có checkpoint, PLY hay video)

In [ ]:
!bash /workspace/SpatialRoom3D/scripts/pod/results.sh
!bash /workspace/SpatialRoom3D/scripts/pod/pack_results.sh

## 11. Benchmark và đánh giá (tùy chọn)

Chọn `BENCH` và `WHAT` rồi chạy; việc dùng GPU chạy nền, xếp hàng sau việc khác. Lượt đã có cùng cấu hình tự bỏ qua. Protocol từng benchmark: `experiments/<benchmark>/README.md`.

| `WHAT` | Việc | Ghi chú |
|---|---|---|
| `"gsplat"` | `default` rồi `mcmc` cùng số Gaussian, bảng so sánh (`scripts/run_train.sh <benchmark> gsplat`) | Tốn GPU. `SEEDS = "42 43 44"` để đủ 3 seed (cần trước khi nói "cải thiện") |
| `"b0"` | 3DGS gốc của Inria (`scripts/run_train.sh <benchmark> b0`) | Tốn GPU. Mốc đối chiếu cho `default` |
| `"evaluate"` | Lập lại bảng so sánh từ kết quả đã có (`scripts/run_evaluate.sh`) | Không dùng GPU, chạy ngay |
| `"prepare"` | Tìm / tải dữ liệu, kiểm tra bố cục (`scripts/prepare_data.sh`) | Không dùng GPU. Mip-NeRF 360 (khoảng 12 GB) chỉ tải khi `DOWNLOAD = True` |

Mip-NeRF 360: `WHAT = "prepare"` phải báo OK trước. Chạy một phần cảnh: `SCENES = "room counter"`.

In [ ]:
BENCH = "deep_blending"   # "deep_blending" hoặc "mipnerf360"
WHAT = "evaluate"         # "gsplat", "b0", "evaluate" hoặc "prepare"
SEEDS = "42"              # vd. "42 43 44"
SCENES = ""               # để trống = mọi cảnh của benchmark
DOWNLOAD = False          # chỉ dùng với WHAT = "prepare": cho phép tải dữ liệu không tự tải (Mip-NeRF 360)

import os, subprocess
repo = "/workspace/SpatialRoom3D"
env = dict(os.environ, SEEDS=SEEDS, DOWNLOAD="1" if DOWNLOAD else "0")
if SCENES.strip():
    env["SCENES"] = SCENES
if WHAT == "evaluate":
    cmd = ["bash", f"{repo}/scripts/run_evaluate.sh", BENCH]
elif WHAT == "prepare":
    cmd = ["bash", f"{repo}/scripts/prepare_data.sh", BENCH]
else:
    cmd = ["bash", f"{repo}/scripts/pod/job.sh", "start", f"{WHAT}_{BENCH}", "--",
           "bash", f"{repo}/scripts/run_train.sh", BENCH, WHAT]
out = subprocess.run(cmd, capture_output=True, text=True, errors="replace", cwd=repo, env=env)
print(out.stdout[-6000:] + out.stderr[-3000:])

## 12. Dừng mọi việc đang chạy

Đặt `DUNG=1` rồi chạy ô: dừng các việc chạy nền (kể cả việc đang xếp hàng) và mọi tiến trình con. Chạy lại ô tương ứng sau đó: phần đã xong tự bỏ qua, lượt train dở được chạy lại từ đầu.

In [ ]:
%%bash
DUNG=0   # 1: dừng mọi việc chạy nền
if [ "$DUNG" != 1 ]; then echo "Chưa dừng gì: đặt DUNG=1 rồi chạy lại ô này"; exit 0; fi
[ -f /workspace/SpatialRoom3D/scripts/pod/job.sh ] && bash /workspace/SpatialRoom3D/scripts/pod/job.sh stop --all
nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

## Khi gặp lỗi

| Thấy trong log / ô | Cách xử lý |
|---|---|
| `✗ GitHub chưa nhận khóa`, `Permission denied (publickey)` (ô 2, 3) | Khóa công khai chưa được thêm vào Deploy keys của **đúng** repo `SpatialRoom3D`: thêm rồi chạy lại ô 2 |
| `Code trên pod có thay đổi chưa commit` (ô 3) | Có file trong `/workspace/SpatialRoom3D` bị sửa (vd. mở notebook trong repo): xem `git -C /workspace/SpatialRoom3D status`, tự quyết định giữ hay bỏ |
| `no Python with PyYAML` | `python3 -m pip install pyyaml`, hoặc chờ ô 4 cài xong env gs-gsplat |
| `driver supports CUDA 12.6 < 12.8` | Ô 4: đặt `TORCH_CUDA_OVERRIDE="cu126"`, chạy lại ô 4 |
| `Killed` khi build gsplat | Ô 4: đặt `MAX_JOBS="4"`, chạy lại ô 4 |
| `compute capability ... is not in TORCH_CUDA_ARCH_LIST` | GPU không được hỗ trợ (vd. T4, RTX 50xx): deploy GPU khác |
| `another job is using the GPU` | Đang có việc khác chạy: xem ô 5; muốn dừng thì ô 12 |
| `MẤT` ở ô 5 | Pod tắt hoặc việc bị giết giữa chừng: chạy lại ô vừa dùng |
| `dataset not found` (ô 7, 11) | Dữ liệu benchmark chưa có ở chỗ `configs/<benchmark>.yaml` ghi: ô 6 hoặc ô 11 với `WHAT = "prepare"` |
| `prepared must be <results.gsplat>/<cảnh>/colmap` | `configs/local.yaml` chỉ đổi một trong hai khóa của `custom_scene`: đổi cả hai |
| `Chỉ có 1 ảnh`, `Chỉ có N ảnh dùng được` | Chụp thêm (ít nhất 10, nên 30–100 ảnh), tải vào cùng thư mục, chạy lại ô 8 |
| `ảnh khác kích thước` | Lẫn ảnh dọc và ngang, hoặc khác ống kính: bỏ các ảnh ở nhóm ít hơn (tên in trong thông báo) hoặc chụp lại cùng chiều máy |
| `không ảnh nào có tiêu cự trong EXIF` | Ảnh đã mất EXIF (gửi qua ứng dụng nhắn tin, chụp màn hình, chỉnh sửa): dùng file gốc của máy nếu có |
| `đã được dựng từ ..., không phải ...` | Tên cảnh đã dùng cho nguồn khác: đặt `SCENE` mới |
| `COLMAP verdict FAIL` | Xem các dòng `gợi ý` ở ô 9, chụp / quay lại theo `docs/data/capture_protocol.md` và chạy ô 8 với tên cảnh mới |
| `góc nhìn: THIẾU` (ô 9), `VIEW COVERAGE: LOW` (log) | Chỉ là cảnh báo: có hướng chưa ảnh nào nhìn tới, hoặc camera gần như đứng yên |
| `not a finished run` (ô 9) | Lượt `RUN` chưa train xong hoặc tên sai: xem tên các lượt trong bảng của ô 9 |
| `COLMAP model ... changed after this run was trained` (ô 9) | Cảnh đã làm lại COLMAP sau lượt này: chạy ô 8 với `SOURCE = "-"` (lượt cũ tự được train lại), rồi render lượt mới; `XUAT_PLY` của lượt cũ vẫn được |
| Lỗi khác | Chạy ô 10 và gửi gói `ket_qua_*.tar.gz` |